# 6.1 Deployment — Alternative Submissions

> **Project:** Titanic Survival Prediction
> **Date:** 2026-03-31
> **CRISP-DM Phase:** 6. Deployment — Task 6.1
>
> **Context:** The tuned XGBoost submission scored **0.76076** on Kaggle — an 8.65% drop from the 84.73% OOF estimate. This falls below both the 80% target (BSC2) and 78% minimum (BSC1). Risk R5 (CV-Kaggle gap) materialized far worse than expected.
>
> This notebook generates alternative submissions with simpler, more regularized models per the fallback plan in 5.3.

## Setup

In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from xgboost import XGBClassifier

PROJECT_ROOT = Path(__file__).resolve().parent.parent if "__file__" in dir() else Path.cwd()
if (PROJECT_ROOT / "notebooks").is_dir():
    pass
elif (PROJECT_ROOT.parent / "notebooks").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "processed"
SUBMISSIONS_DIR = PROJECT_ROOT / "submissions"
SUBMISSIONS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
N_FOLDS = 5
cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

# Load prepared data
train_df = pd.read_csv(DATA_DIR / "train_formatted.csv")
test_df = pd.read_csv(DATA_DIR / "test_formatted.csv")

X_train = train_df.drop(columns=["Survived", "PassengerId"])
y_train = train_df["Survived"]
X_test = test_df.drop(columns=["PassengerId"])
test_ids = test_df["PassengerId"]

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Features: {X_train.columns.tolist()}")

Train: (891, 26), Test: (418, 26)
Features: ['Pclass', 'Sex', 'Age', 'Fare', 'FareLog', 'SibSp', 'Parch', 'FamilySize', 'FamilySizeBin', 'IsAlone', 'HasCabin', 'TicketGroupSize', 'AgeMissing', 'Deck_A', 'Deck_B', 'Deck_C', 'Deck_D', 'Deck_E', 'Deck_F', 'Deck_G', 'Deck_T', 'Embarked_Q', 'Embarked_S', 'Title_Master', 'Title_Miss', 'Title_Mrs']


## Diagnosis: Why Did XGBoost Score 76% on Kaggle?

The tuned XGBoost had a ~7% train-validation gap (92% train vs 84.73% OOF). On Kaggle it dropped further to 76.08%. Possible causes:
1. **Overfitting** — `max_depth=6` on 891 rows is too complex; the model memorized training noise
2. **Optimistic CV estimate** — hyperparameters were selected on the same CV folds used to report accuracy
3. **Feature overfitting** — 26 features for 891 rows is a high ratio; some engineered features may add noise

**Strategy:** Train progressively simpler models and compare. If simpler models also drop sharply, the issue is in the features/data. If they hold up, the issue is model complexity.

In [2]:
def evaluate_and_submit(model, X_tr, y_tr, X_te, ids, name, cv=cv):
    """Evaluate via CV, train on full data, generate submission."""
    scores = cross_val_score(model, X_tr, y_tr, cv=cv, scoring="accuracy")
    print(f"\n{'='*60}")
    print(f"{name}")
    print(f"  CV Accuracy: {scores.mean():.4f} ± {scores.std():.4f}")
    print(f"  Fold scores: {[f'{s:.4f}' for s in scores]}")
    
    # Train on full training set
    model.fit(X_tr, y_tr)
    
    # Check train accuracy to measure overfitting
    train_acc = model.score(X_tr, y_tr) if hasattr(model, "score") else None
    if train_acc:
        print(f"  Train Accuracy: {train_acc:.4f}")
        print(f"  Overfit gap: {train_acc - scores.mean():.4f}")
    
    # Generate predictions
    preds = model.predict(X_te)
    submission = pd.DataFrame({"PassengerId": ids, "Survived": preds.astype(int)})
    
    path = SUBMISSIONS_DIR / f"submission_{name.lower().replace(' ', '_')}.csv"
    submission.to_csv(path, index=False)
    print(f"  Saved: {path.name}")
    print(f"  Prediction dist: {dict(pd.Series(preds).value_counts().sort_index())}")
    
    return scores.mean(), scores.std(), train_acc

## Model 1: Logistic Regression (strong regularization)

Simplest ML model. Low variance, high bias — should have the smallest CV-to-Kaggle gap.

In [3]:
results = {}

lr = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(C=0.5, penalty="l2", solver="lbfgs", max_iter=1000, random_state=RANDOM_STATE))
])
cv_mean, cv_std, train_acc = evaluate_and_submit(lr, X_train, y_train, X_test, test_ids, "logistic_regression")
results["Logistic Regression"] = {"cv": cv_mean, "std": cv_std, "train": train_acc}


logistic_regression
  CV Accuracy: 0.8283 ± 0.0214
  Fold scores: ['0.8547', '0.8371', '0.7978', '0.8090', '0.8427']
  Train Accuracy: 0.8373
  Overfit gap: 0.0090
  Saved: submission_logistic_regression.csv
  Prediction dist: {0: np.int64(248), 1: np.int64(170)}


## Model 2: Random Forest (heavily regularized)

Stronger regularization than the 4.3 tuned RF: shallower trees, more minimum samples per leaf.

In [4]:
rf = RandomForestClassifier(
    n_estimators=200, max_depth=5, min_samples_leaf=10,
    max_features="sqrt", random_state=RANDOM_STATE, n_jobs=-1
)
cv_mean, cv_std, train_acc = evaluate_and_submit(rf, X_train, y_train, X_test, test_ids, "random_forest_regularized")
results["Random Forest (reg)"] = {"cv": cv_mean, "std": cv_std, "train": train_acc}


random_forest_regularized
  CV Accuracy: 0.8249 ± 0.0139
  Fold scores: ['0.8212', '0.8146', '0.8090', '0.8315', '0.8483']
  Train Accuracy: 0.8429
  Overfit gap: 0.0180
  Saved: submission_random_forest_regularized.csv
  Prediction dist: {0: np.int64(272), 1: np.int64(146)}


## Model 3: XGBoost (heavily regularized)

Much shallower trees and stronger regularization than the 4.3 version to reduce overfitting.

In [5]:
xgb = XGBClassifier(
    n_estimators=100, max_depth=3, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.6,
    reg_alpha=1.0, reg_lambda=5.0, min_child_weight=5,
    random_state=RANDOM_STATE, eval_metric="logloss",
    use_label_encoder=False, n_jobs=-1
)
cv_mean, cv_std, train_acc = evaluate_and_submit(xgb, X_train, y_train, X_test, test_ids, "xgboost_regularized")
results["XGBoost (reg)"] = {"cv": cv_mean, "std": cv_std, "train": train_acc}


xgboost_regularized
  CV Accuracy: 0.8316 ± 0.0160
  Fold scores: ['0.8436', '0.8258', '0.8034', '0.8371', '0.8483']
  Train Accuracy: 0.8552
  Overfit gap: 0.0236
  Saved: submission_xgboost_regularized.csv
  Prediction dist: {0: np.int64(273), 1: np.int64(145)}


## Model 4: Voting Ensemble (LR + RF + XGBoost)

Majority vote of the three models above. Diversity should reduce variance.

In [6]:
ensemble = VotingClassifier(
    estimators=[
        ("lr", Pipeline([
            ("scaler", StandardScaler()),
            ("lr", LogisticRegression(C=0.5, penalty="l2", solver="lbfgs", max_iter=1000, random_state=RANDOM_STATE))
        ])),
        ("rf", RandomForestClassifier(
            n_estimators=200, max_depth=5, min_samples_leaf=10,
            max_features="sqrt", random_state=RANDOM_STATE, n_jobs=-1
        )),
        ("xgb", XGBClassifier(
            n_estimators=100, max_depth=3, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.6,
            reg_alpha=1.0, reg_lambda=5.0, min_child_weight=5,
            random_state=RANDOM_STATE, eval_metric="logloss",
            use_label_encoder=False, n_jobs=-1
        )),
    ],
    voting="hard"
)
cv_mean, cv_std, train_acc = evaluate_and_submit(ensemble, X_train, y_train, X_test, test_ids, "voting_ensemble")
results["Voting Ensemble"] = {"cv": cv_mean, "std": cv_std, "train": train_acc}


voting_ensemble
  CV Accuracy: 0.8339 ± 0.0147
  Fold scores: ['0.8492', '0.8258', '0.8090', '0.8427', '0.8427']


  Train Accuracy: 0.8530
  Overfit gap: 0.0191
  Saved: submission_voting_ensemble.csv
  Prediction dist: {0: np.int64(271), 1: np.int64(147)}


## Model 5: Minimal features — Sex + Pclass + Title only

Test whether the 26-feature set is itself causing overfitting. Use only the 3 strongest predictors.

In [7]:
minimal_features = ["Sex", "Pclass", "Title_Master", "Title_Miss", "Title_Mrs"]
X_train_minimal = X_train[minimal_features]
X_test_minimal = X_test[minimal_features]

lr_minimal = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(C=1.0, penalty="l2", solver="lbfgs", max_iter=1000, random_state=RANDOM_STATE))
])
cv_mean, cv_std, train_acc = evaluate_and_submit(
    lr_minimal, X_train_minimal, y_train, X_test_minimal, test_ids, "lr_minimal_features"
)
results["LR (minimal)"] = {"cv": cv_mean, "std": cv_std, "train": train_acc}


lr_minimal_features
  CV Accuracy: 0.7946 ± 0.0166
  Fold scores: ['0.7989', '0.7865', '0.7978', '0.7697', '0.8202']
  Train Accuracy: 0.8002
  Overfit gap: 0.0056
  Saved: submission_lr_minimal_features.csv
  Prediction dist: {0: np.int64(262), 1: np.int64(156)}


## Comparison Summary

In [8]:
summary = pd.DataFrame(results).T
summary["overfit_gap"] = summary["train"] - summary["cv"]
summary = summary.round(4)

# Add the original XGBoost for reference
summary.loc["XGBoost (4.3 tuned)"] = {"cv": 0.8473, "std": 0.0169, "train": 0.92, "overfit_gap": 0.0727}
summary.loc["XGBoost (4.3 tuned)", "kaggle"] = 0.76076

print("=" * 70)
print("MODEL COMPARISON — CV vs Overfitting")
print("=" * 70)
print(summary.to_string())
print("\nNote: Models with smaller overfit gaps are more likely to hold up on Kaggle.")
print("The original XGBoost (4.3) had the largest gap and scored 0.76076 on Kaggle.")
print("\nSubmission files generated in submissions/ — submit the model with the smallest overfit gap first.")

MODEL COMPARISON — CV vs Overfitting
                         cv     std   train  overfit_gap   kaggle
Logistic Regression  0.8283  0.0214  0.8373       0.0090      NaN
Random Forest (reg)  0.8249  0.0139  0.8429       0.0180      NaN
XGBoost (reg)        0.8316  0.0160  0.8552       0.0236      NaN
Voting Ensemble      0.8339  0.0147  0.8530       0.0191      NaN
LR (minimal)         0.7946  0.0166  0.8002       0.0056      NaN
XGBoost (4.3 tuned)  0.8473  0.0169  0.9200       0.0727  0.76076

Note: Models with smaller overfit gaps are more likely to hold up on Kaggle.
The original XGBoost (4.3) had the largest gap and scored 0.76076 on Kaggle.

Submission files generated in submissions/ — submit the model with the smallest overfit gap first.


## Conclusions

**Problem:** The tuned XGBoost from 4.3 scored 0.76076 on Kaggle — an 8.65% drop from the 84.73% OOF estimate. This triggered the fallback plan from 5.3.

**Submissions generated:**
1. `submission_logistic_regression.csv` — simplest model, lowest variance
2. `submission_random_forest_regularized.csv` — heavily regularized RF
3. `submission_xgboost_regularized.csv` — XGBoost with max_depth=3 and strong L1/L2
4. `submission_voting_ensemble.csv` — majority vote of LR + RF + XGBoost
5. `submission_lr_minimal_features.csv` — LR with only Sex, Pclass, Title (diagnostic)

**Recommended submission order:** Start with the model showing the smallest overfit gap (train - CV). If Logistic Regression holds up well on Kaggle relative to its CV, the problem was model complexity. If even the minimal-feature LR drops sharply, the issue may be in the feature engineering pipeline.

**Next steps:**
- Submit each file to Kaggle and record scores
- Update the comparison table with actual Kaggle scores
- Update 6.3 final report with revised results
- If no model exceeds 80%, revisit Phase 3 (feature engineering)